In [2]:
# import libraries
import os
import numpy as np
import cv2
import joblib
from google.colab.patches import cv2_imshow
from tensorflow.keras.models import load_model

IMG_SIZE = (224, 224)


In [3]:
# Mount drive and all save model
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Machine_Learning/Research_base_Model"

# Load deep learning models
cnn_model = load_model(f"{BASE}/final_cnn_model.h5")
feature_model = load_model(f"{BASE}/feature_extractor.h5")

# Load ML models
rf = joblib.load(f"{BASE}/cnn_rf_model.pkl")
svm = joblib.load(f"{BASE}/cnn_svm_model.pkl")
knn = joblib.load(f"{BASE}/cnn_knn_model.pkl")

# Load scaler
scaler = joblib.load(f"{BASE}/feature_scaler.pkl")

print("🔥 All models loaded successfully!")


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.5.2 when using version 1.6.1. This might lead to breaking cod

🔥 All models loaded successfully!


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
# Load Test Image
TEST_DIR = f"{BASE}/chest_xray_dataset/test"

test_files = []

for label in ["NORMAL", "PNEUMONIA"]:
    folder = os.path.join(TEST_DIR, label)
    for f in os.listdir(folder):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            test_files.append({
                "path": os.path.join(folder, f),
                "label": label
            })

print("Total test images found:", len(test_files))


Total test images found: 624


In [5]:
# Simple prediction function
def predict_image_simple(path):
    print("Image path:", path)

    # Load & display
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cv2_imshow(img_rgb)

    # Preprocess
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    x = img_resized.astype("float32") / 255.0
    x = np.expand_dims(x, axis=0)

    # CNN prediction
    cnn_prob = cnn_model.predict(x)[0][0]
    cnn_pred = 1 if cnn_prob >= 0.5 else 0

    # Feature
    feat = feature_model.predict(x)

    # RF
    rf_prob = rf.predict_proba(feat)[0][1]
    rf_pred = 1 if rf_prob >= 0.5 else 0

    # Scaled features
    feat_scaled = scaler.transform(feat)

    # SVM
    svm_prob = svm.predict_proba(feat_scaled)[0][1]
    svm_pred = 1 if svm_prob >= 0.5 else 0

    # KNN
    knn_prob = knn.predict_proba(feat_scaled)[0][1]
    knn_pred = 1 if knn_prob >= 0.5 else 0

    # Majority vote
    preds = [cnn_pred, rf_pred, svm_pred, knn_pred]
    final = 1 if preds.count(1) >= 2 else 0

    print("\n===== Predictions =====")
    print(f"CNN : {'PNEUMONIA' if cnn_pred else 'NORMAL'} (prob={cnn_prob:.4f})")
    print(f"RF  : {'PNEUMONIA' if rf_pred else 'NORMAL'} (prob={rf_prob:.4f})")
    print(f"SVM : {'PNEUMONIA' if svm_pred else 'NORMAL'} (prob={svm_prob:.4f})")
    print(f"KNN : {'PNEUMONIA' if knn_pred else 'NORMAL'} (prob={knn_prob:.4f})")

    print("\nFinal Output:",
          "🔥 PNEUMONIA DETECTED" if final else "✅ NORMAL")

    return final


In [ ]:
# Test prediction
index = 0   # you can change this to test any image
info = test_files[index]

print("True Label =", info["label"])
predict_image_simple(info["path"])
